# 🌍 Good Vibes Workshop — ECCB 2026

## Responsible use of AI for large-scale, rapid Earth Observation workflows

---

**Authors:** AB, HFT, HH, JW  
**Affiliation:** Institute of Zoology, Zoological Society of London (ZSL)  
**Conference:** ECCB 2026  
**Duration:** ~90 minutes hands-on session

---

### Workshop Overview

In this workshop, you'll learn how to **responsibly** use AI coding assistants (like Gemini, ChatGPT, Copilot) to accelerate Earth Observation (EO) workflows using **Google Earth Engine (GEE)**. We'll cover:

1. **Visualising** satellite imagery interactively
2. **Analysing** deforestation patterns at scale
3. **Debugging** AI-generated code (because AI makes mistakes!)
4. **Building** your own analyses with AI assistance

### ⚠️ Key Philosophy

> *"AI is a powerful accelerator, but you are the scientist. You must understand, validate, and take responsibility for every line of code that goes into your analysis."*

---

📄 **Licence:** This notebook is released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).  
You are free to share and adapt this material, provided you give appropriate credit.

---

---
# Section 0: Setup & Authentication

Before we begin, we need to:
1. Install the required Python packages
2. Authenticate with Google Earth Engine
3. Verify everything works

**If you're running this on Google Colab**, most dependencies are pre-installed, but we'll update them to ensure compatibility.

In [ ]:
# ============================================================
# STEP 0a: Install required packages
# ============================================================
# geemap provides interactive map widgets for Google Earth Engine
# earthengine-api is the Python client for GEE
# The -q flag suppresses verbose install output

!pip install -q geemap earthengine-api

In [ ]:
# ============================================================
# STEP 0b: Import all required libraries
# ============================================================

import ee                          # Google Earth Engine Python API
import geemap                      # Interactive mapping for GEE
import datetime                    # For timestamps in reproducibility metadata
import hashlib                     # For hashing notebook content
import json                        # For reading notebook JSON
import sys                         # For Python version info
import matplotlib.pyplot as plt    # For plotting charts
import numpy as np                 # For numerical operations

print("✅ All libraries imported successfully!")

### 🔑 Authentication

Google Earth Engine requires authentication. When you run the cell below:

1. **`ee.Authenticate()`** will open a browser window asking you to sign in with your Google account and grant access to Earth Engine.
2. **`ee.Initialize()`** connects your session to the GEE servers.

**⚠️ IMPORTANT:** You **must** replace `'YOUR_PROJECT_ID'` with your own Google Cloud Project ID that has the Earth Engine API enabled. If you don't have one, follow the instructions at [signup.earthengine.google.com](https://signup.earthengine.google.com/).

In [ ]:
# ============================================================
# STEP 0c: Authenticate and initialise Earth Engine
# ============================================================

# Step 1: Authenticate — this opens a browser window for you to sign in
ee.Authenticate()

# Step 2: Initialise — connect to GEE servers with your project
# ⚠️ REPLACE 'YOUR_PROJECT_ID' with your actual Google Cloud Project ID!
# Example: ee.Initialize(project='my-gee-project-12345')
ee.Initialize(project='YOUR_PROJECT_ID')

print("✅ Earth Engine authenticated and initialised!")

In [ ]:
# ============================================================
# STEP 0d: Verification — load a simple image to test everything
# ============================================================
# We'll load the SRTM Digital Elevation Model — a simple, reliable dataset
# that's always available. If this works, you're good to go!

dem = ee.Image('USGS/SRTMGL1_003')

# Get some basic info about the image
# .getInfo() sends the request to GEE servers and returns the result
elevation_info = dem.select('elevation').reduceRegion(
    reducer=ee.Reducer.minMax(),
    geometry=ee.Geometry.Rectangle([116.0, 5.5, 117.0, 6.5]),  # Small area in Sabah
    scale=1000,      # 1km resolution for a quick test
    bestEffort=True   # Allows GEE to adjust if needed
).getInfo()

print("🏔️  Test region elevation range:")
print(f"   Min: {elevation_info['elevation_min']:.0f} m")
print(f"   Max: {elevation_info['elevation_max']:.0f} m")
print("\n✅ GEE connection verified — you're ready for the workshop!")

---
# Section 1: Reproducibility Metadata

### ⚠️ Teaching Point: Why Reproducibility Matters

In scientific research, **reproducibility is non-negotiable**. When using AI to write code and GEE to process data, there are several moving parts that can change over time:

- **Software versions** may change (new features, bug fixes, deprecations)
- **Datasets** on GEE may be updated (new data added, corrections applied)
- **AI models** evolve — the same prompt may produce different code tomorrow

By recording metadata at the start of every analysis, you create a **provenance record** that allows you (or others) to understand exactly what was used.

This is especially important when using AI assistants: the AI model version, the prompt used, and the code generated should all be documented.

In [ ]:
# ============================================================
# STEP 1: Auto-capture reproducibility metadata
# ============================================================
# This cell records the exact software versions and dataset IDs
# used in this analysis. Run this at the START of every session.

# --- Software versions ---
python_version = sys.version
ee_version = ee.__version__
geemap_version = geemap.__version__

# --- Dataset identifiers ---
# The Hansen Global Forest Change dataset is versioned — always record which version!
HANSEN_DATASET_ID = 'UMD/hansen/global_forest_change_2023_v1_11'

# --- Timestamp ---
run_timestamp = datetime.datetime.now().strftime('%Y-%m-%d %H:%M:%S UTC')

# --- Notebook hash (for version tracking) ---
# This computes a SHA-256 hash of the notebook file itself.
# If any cell changes, the hash changes — so you know if the code was modified.
try:
    notebook_path = 'Good_Vibes_Workshop.ipynb'  # Adjust path if needed
    with open(notebook_path, 'rb') as f:
        notebook_hash = hashlib.sha256(f.read()).hexdigest()[:16]
except FileNotFoundError:
    notebook_hash = 'N/A (running in Colab — hash available when saved locally)'

# --- Print the provenance block ---
print("=" * 60)
print("📋 REPRODUCIBILITY METADATA")
print("=" * 60)
print(f"Run timestamp:       {run_timestamp}")
print(f"Python version:      {python_version.split()[0]}")
print(f"ee version:          {ee_version}")
print(f"geemap version:      {geemap_version}")
print(f"Hansen GFC dataset:  {HANSEN_DATASET_ID}")
print(f"Notebook hash:       {notebook_hash}")
print("=" * 60)
print("\n💡 TIP: Copy this block into your methods section!")

### 📝 What to record in your methods section

When you publish results from GEE analyses, include:

| Element | Example |
|---------|--------|
| **Dataset ID & version** | `UMD/hansen/global_forest_change_2023_v1_11` |
| **Software versions** | Python 3.10, earthengine-api 0.1.x, geemap 0.x.x |
| **Analysis date** | 2026-06-29 |
| **AI assistant used** | e.g., "Code generated with Gemini 2.5, manually reviewed" |
| **Code availability** | Link to GitHub/Zenodo with the exact notebook |
| **Parameters** | Scale, thresholds, date ranges, region bounds |

---

---
# Section 2: Worked Example 1 — Visualise Satellite Imagery
**⏱️ ~10 minutes**

---

### What is satellite imagery?

Satellites like **Landsat 8/9** capture images of the Earth's surface in multiple **spectral bands** — not just the red, green, and blue light that our eyes see, but also **near-infrared (NIR)**, **shortwave infrared (SWIR)**, and other wavelengths.

Different band combinations reveal different features:

| Composite | Bands (Landsat 8/9 SR) | What it shows |
|-----------|----------------------|---------------|
| **True colour** | B4, B3, B2 (Red, Green, Blue) | What the human eye would see |
| **False colour** | B5, B4, B3 (NIR, Red, Green) | Vegetation appears bright red |
| **SWIR composite** | B7, B5, B4 (SWIR2, NIR, Red) | Burn scars, geology, moisture |

### What is compositing?

Satellites pass over the same location every few days/weeks. A single image may have clouds. By **compositing** (taking the median pixel value over a time period), we can create cloud-free mosaics.

### Our study area: Sabah, Malaysian Borneo

Sabah is a biodiversity hotspot with extensive tropical forests, but has experienced significant deforestation due to palm oil expansion. It's an ideal region for studying forest change.

In [ ]:
# ============================================================
# STEP 2a: Define the study area
# ============================================================
# Sabah, Malaysian Borneo — a biodiversity hotspot
# Coordinates: roughly 115.5°E to 118.5°E, 4.5°N to 7°N

sabah_bbox = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# 🔧 GEE Best Practice: Define your region of interest FIRST,
# then use it for filtering. This keeps everything consistent.

print(f"Study area defined: Sabah, Malaysian Borneo")
print(f"Bounding box: [115.5°E, 4.5°N] to [118.5°E, 7.0°N]")

In [ ]:
# ============================================================
# STEP 2b: Load and composite Landsat 8/9 Surface Reflectance
# ============================================================

# 🔧 GEE Best Practice: Filter EARLY with filterBounds() and filterDate()
# This tells GEE to only load images that intersect our study area
# and fall within our date range — massively reducing computation.

# 🔧 GEE Best Practice: Use .select() EARLY to reduce memory.
# We only need the optical bands, so we select them before compositing.

# Load Landsat 8 Collection 2, Tier 1, Surface Reflectance
landsat = (
    ee.ImageCollection('LANDSAT/LC08/C02/T1_L2')
    .filterBounds(sabah_bbox)           # ← Filter spatially FIRST
    .filterDate('2023-01-01', '2023-12-31')  # ← Then filter temporally
    .select(['SR_B2', 'SR_B3', 'SR_B4', 'SR_B5'])  # ← Select only needed bands
    .filter(ee.Filter.lt('CLOUD_COVER', 30))  # ← Remove very cloudy scenes
)

# Create a median composite — this removes clouds and noise
# by taking the middle value for each pixel across all images
composite = landsat.median()

# Landsat Collection 2 SR data needs scaling:
# Surface reflectance bands have a scale factor of 0.0000275 and offset of -0.2
composite_scaled = composite.multiply(0.0000275).add(-0.2)

print(f"Loaded {landsat.size().getInfo()} Landsat 8 scenes for 2023")
print("Median composite created successfully!")

In [ ]:
# ============================================================
# STEP 2c: Create an interactive map with satellite imagery
# ============================================================

# Create a geemap Map object centred on Sabah
Map1 = geemap.Map(center=[5.8, 117.0], zoom=8)

# --- True-colour composite ---
# Bands B4 (Red), B3 (Green), B2 (Blue) = what the human eye sees
true_colour_vis = {
    'bands': ['SR_B4', 'SR_B3', 'SR_B2'],  # Red, Green, Blue
    'min': 0.0,
    'max': 0.3,   # Reflectance values typically range 0–0.3
}
Map1.addLayer(composite_scaled, true_colour_vis, 'True Colour (RGB)')

# --- False-colour composite ---
# Bands B5 (NIR), B4 (Red), B3 (Green)
# Vegetation strongly reflects NIR, so forests appear BRIGHT RED
# This makes it easy to distinguish forest from non-forest
false_colour_vis = {
    'bands': ['SR_B5', 'SR_B4', 'SR_B3'],  # NIR, Red, Green
    'min': 0.0,
    'max': 0.4,
}
Map1.addLayer(composite_scaled, false_colour_vis, 'False Colour (NIR)', shown=False)

print("Map created! Scroll down to interact with it.")
print("💡 TIP: Use the layer control (top right) to toggle between true and false colour.")

In [ ]:
# ============================================================
# STEP 2d: Add a Protected Area boundary from WDPA
# ============================================================
# The World Database on Protected Areas (WDPA) is available on GEE.
# We'll add the Danum Valley Conservation Area — one of the most
# important primary rainforest reserves in Southeast Asia.

# 🔧 GEE Best Practice: Filter the FeatureCollection before using it.
# WDPA contains ~270,000 protected areas globally — we only need one!

wdpa = ee.FeatureCollection('WCMC/WDPA/current/polygons')

# Filter to protected areas in Sabah
# We search by name for Danum Valley
danum_valley = wdpa.filter(
    ee.Filter.stringContains('NAME', 'Danum Valley')
)

# Style the protected area boundary
pa_style = {'color': '00FF00', 'fillColor': '00000000', 'width': 2}
Map1.addLayer(danum_valley.style(**pa_style), {}, 'Danum Valley CA')

# Also show all Sabah PAs for context
sabah_pas = wdpa.filterBounds(sabah_bbox)
pa_style_all = {'color': 'FFFFFF80', 'fillColor': '00000000', 'width': 1}
Map1.addLayer(sabah_pas.style(**pa_style_all), {}, 'All PAs (Sabah)', shown=False)

# Display the map
Map1

### ⚠️ Teaching Point

**You've just visualised satellite data for an entire region in seconds!**

What just happened behind the scenes:
- GEE loaded **hundreds of Landsat scenes** from its petabyte archive
- It composited them into a cloud-free mosaic
- It rendered the result as interactive map tiles
- All processing happened on Google's servers — nothing was downloaded to your computer

This is the power of cloud-based EO: you can explore data at continental scales without downloading a single file.

**With AI assistance, the code to do this can be generated in seconds. But you need to understand what each line does** — otherwise, how do you know the colours are correct? The band assignments are right? The date range is appropriate?

---

---
# Section 3: Worked Example 2 — Deforestation Analysis
**⏱️ ~15 minutes**

---

### The Hansen Global Forest Change Dataset

The **Hansen et al. (2013)** Global Forest Change (GFC) dataset is one of the most widely used products for monitoring deforestation worldwide. It's derived from Landsat imagery and provides:

| Band | Description | Values |
|------|-------------|--------|
| `treecover2000` | Tree canopy cover in year 2000 | 0–100 (percentage) |
| `loss` | Forest loss during 2001–2023 | 0 (no loss) or 1 (loss) |
| `gain` | Forest gain during 2001–2012 | 0 (no gain) or 1 (gain) |
| `lossyear` | Year of forest loss | 1–23 (= years 2001–2023) |
| `datamask` | Data availability | 0 (no data), 1 (land), 2 (water) |

**Key details:**
- **Resolution:** 30 metres (approximately)
- **Coverage:** Global, between 80°N and 60°S latitude
- **Forest definition:** Tree canopy cover > 30% is commonly used as a threshold
- **Citation:** Hansen, M.C. et al. (2013) *Science*, 342(6160)

### What we'll calculate

1. Where was there forest in 2000?
2. Where has forest been lost since 2001?
3. How much forest has been lost each year?
4. What's the temporal trend?

In [ ]:
# ============================================================
# STEP 3a: Load the Hansen Global Forest Change dataset
# ============================================================

# 🔧 GEE Best Practice: Use .select() EARLY to reduce memory.
# The Hansen dataset has multiple bands — we only need a few.

gfc = ee.Image(HANSEN_DATASET_ID).select(
    ['treecover2000', 'loss', 'lossyear']
)

# 🔧 GEE Best Practice: NO .clip() here!
# We'll use the region parameter when exporting or reducing.
# .clip() adds unnecessary per-pixel geometry intersection overhead.

print(f"Loaded Hansen GFC: {HANSEN_DATASET_ID}")
print(f"Selected bands: treecover2000, loss, lossyear")

In [ ]:
# ============================================================
# STEP 3b: Define forest masks and loss layers
# ============================================================

# --- Forest in year 2000 ---
# We use a threshold of 30% canopy cover to define "forest"
# This is a common threshold, but different ecosystems may need different values
FOREST_THRESHOLD = 30  # percent canopy cover

treecover2000 = gfc.select('treecover2000')
forest_2000 = treecover2000.gte(FOREST_THRESHOLD)  # Binary mask: 1=forest, 0=not

# --- Forest loss (binary: any loss during 2001-2023) ---
loss = gfc.select('loss')

# --- Loss by year (values 1-23 = years 2001-2023) ---
lossyear = gfc.select('lossyear')

# --- Forest loss area ---
# ee.Image.pixelArea() returns area in square metres for each pixel
# We multiply by the loss mask to get area only where loss occurred
# Then convert to hectares (1 hectare = 10,000 m²)
loss_area = loss.multiply(ee.Image.pixelArea()).divide(10000)

print(f"Forest threshold: {FOREST_THRESHOLD}% canopy cover")
print("Forest masks and loss layers created.")

In [ ]:
# ============================================================
# STEP 3c: Visualise forest cover and loss on the map
# ============================================================

# Create a new map for the deforestation analysis
Map2 = geemap.Map(center=[5.8, 117.0], zoom=8)

# --- Layer 1: Tree cover in 2000 ---
# Green shading shows canopy cover density (darker = denser)
Map2.addLayer(
    treecover2000,
    {'min': 0, 'max': 100, 'palette': ['black', 'green']},
    'Tree Cover 2000 (%)',
    shown=False
)

# --- Layer 2: Forest mask (binary: >= 30% cover) ---
# Bright green = forest in 2000
Map2.addLayer(
    forest_2000.selfMask(),  # selfMask() hides 0 values (non-forest)
    {'palette': ['00FF00']},
    'Forest 2000 (≥30%)',
    opacity=0.5
)

# --- Layer 3: Forest loss (red) ---
Map2.addLayer(
    loss.selfMask(),
    {'palette': ['FF0000']},
    'Forest Loss (2001-2023)'
)

# --- Layer 4: Loss by year (colour-ramped) ---
# Cool colours (blue) = early years, warm colours (red) = recent years
Map2.addLayer(
    lossyear.selfMask(),
    {
        'min': 1,
        'max': 23,
        'palette': [
            '0000FF',  # 2001 — blue
            '0080FF',
            '00FFFF',  # cyan
            '00FF80',
            '00FF00',  # green
            '80FF00',
            'FFFF00',  # yellow
            'FF8000',
            'FF0000',  # 2023 — red
        ]
    },
    'Loss Year (2001-2023)',
    shown=False
)

# Display the map
Map2

In [ ]:
# ============================================================
# STEP 3d: Calculate annual deforestation statistics
# ============================================================
# We'll calculate the total area of forest loss for each year
# from 2001 to 2023 within Sabah.
#
# Strategy: For each year, create a mask where lossyear == year,
# multiply by pixel area, and sum across the region.

# 🔧 GEE Best Practice: Use reduceRegion with the geometry parameter
# instead of clipping the image first.

years = list(range(1, 24))  # 1-23 = 2001-2023
year_labels = [str(2000 + y) for y in years]
annual_loss_ha = []

print("Calculating annual deforestation (this may take a minute)...")
print()

for year_val in years:
    # Create a mask for this specific year
    year_mask = lossyear.eq(year_val)

    # Calculate area of loss in this year (in hectares)
    year_loss_area = year_mask.multiply(ee.Image.pixelArea()).divide(10000)

    # Sum the area across Sabah
    stats = year_loss_area.reduceRegion(
        reducer=ee.Reducer.sum(),
        geometry=sabah_bbox,  # ← Region parameter handles spatial extent
        scale=30,             # ← Hansen native resolution is ~30m
        maxPixels=1e13,       # ← Allow large computations
        bestEffort=True       # ← Fall back gracefully if needed
    )

    # Get the value (returns a dictionary; the key is the band name)
    loss_ha = stats.getInfo()['lossyear']
    annual_loss_ha.append(loss_ha)
    print(f"  {2000 + year_val}: {loss_ha:>12,.0f} hectares")

print()
print(f"✅ Total forest loss (2001-2023): {sum(annual_loss_ha):,.0f} hectares")

In [ ]:
# ============================================================
# STEP 3e: Plot annual deforestation as a bar chart
# ============================================================

fig, ax = plt.subplots(figsize=(14, 6))

# Create bars with a colour gradient (light to dark red)
colours = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(years)))
bars = ax.bar(year_labels, annual_loss_ha, color=colours, edgecolor='darkred', linewidth=0.5)

# Add a horizontal line for the mean
mean_loss = np.mean(annual_loss_ha)
ax.axhline(y=mean_loss, color='black', linestyle='--', linewidth=1, label=f'Mean: {mean_loss:,.0f} ha/yr')

# Formatting
ax.set_xlabel('Year', fontsize=12)
ax.set_ylabel('Forest Loss (hectares)', fontsize=12)
ax.set_title('Annual Forest Loss in Sabah, Malaysian Borneo (2001–2023)\nSource: Hansen et al. GFC v1.11', fontsize=14)
ax.legend(fontsize=11)
ax.tick_params(axis='x', rotation=45)

# Add gridlines for readability
ax.grid(axis='y', alpha=0.3)
ax.set_axisbelow(True)

# Format y-axis with commas for thousands
ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))

plt.tight_layout()
plt.show()

print(f"\n📊 Summary Statistics:")
print(f"   Total loss (2001-2023):   {sum(annual_loss_ha):>12,.0f} hectares")
print(f"   Mean annual loss:         {mean_loss:>12,.0f} hectares/year")
print(f"   Peak year:                {year_labels[np.argmax(annual_loss_ha)]} ({max(annual_loss_ha):,.0f} ha)")
print(f"   Lowest year:              {year_labels[np.argmin(annual_loss_ha)]} ({min(annual_loss_ha):,.0f} ha)")

### ⚠️ Teaching Point

**In about 15 minutes, you've done an analysis that would have taken days using traditional methods!**

Think about what happened:
- You loaded a **global** dataset covering all forests on Earth
- You filtered to your region of interest
- You calculated pixel-level statistics at **30-metre resolution**
- You generated a 23-year time series of deforestation
- You visualised everything interactively

### 🔧 GEE Best Practices Used in This Section

| Practice | Why it matters |
|----------|---------------|
| `.select()` early | Reduced memory by loading only needed bands |
| No `.clip()` | Used `geometry` parameter in `reduceRegion` instead |
| `.filterBounds()` + `.filterDate()` | Told GEE to only process relevant data |
| `scale=30` in `reduceRegion` | Matched the native resolution of Hansen (30m) |
| `bestEffort=True` | Graceful fallback if computation is too large |

---

---
# Section 4: Worked Example 3 — Find the Bug! 🐛
**⏱️ ~10 minutes**

---

### AI-generated code often contains subtle bugs

One of the biggest risks of using AI coding assistants is that they generate code that **looks correct** and **runs without errors** but produces **wrong results**. These silent failures are far more dangerous than syntax errors.

Below is a GEE script that analyses forest loss in a region. **It was generated by an AI assistant and contains 5 bugs.** The code runs without raising any Python errors — but the results are scientifically wrong.

**Your challenge:** Find all 5 bugs! Work with a partner if you like.

**Hints will be provided below the code — try without them first!**

In [ ]:
# ============================================================
# 🐛 BUGGY CODE — AI-generated forest loss analysis
# ============================================================
# This code was "generated by an AI assistant" to analyse
# forest loss in Sabah between 2010 and 2020.
#
# ⚠️ THERE ARE 5 BUGS. The code RUNS but gives WRONG RESULTS.
# Can you find them all?
# ============================================================

# Load the Hansen dataset
buggy_gfc = ee.Image('UMD/hansen/global_forest_change_2023_v1_11')

# Define the study region
buggy_region = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# === BUG 1 is here ===
# Get tree cover in year 2000
buggy_treecover = buggy_gfc.select('tree_cover')  # 🐛 Bug 1

# === BUG 2 is here ===
# Get the loss year band
buggy_lossyear = buggy_gfc.select('lossyear')

# Filter to forest loss between 2010 and 2020
# Remember: lossyear values 1-23 correspond to years 2001-2023
# So 2010 = value 10, 2020 = value 20

# === BUG 3 is here ===
buggy_loss_decade = buggy_lossyear.gte(10).And(buggy_lossyear.lte(19))  # 🐛 Bug 3

# Calculate the average tree cover percentage in loss areas
# === BUG 4 is here ===
buggy_avg_cover = buggy_treecover.reduceRegion(
    reducer=ee.Reducer.sum(),         # 🐛 Bug 4
    geometry=buggy_region,
    scale=1000,                       # 🐛 Bug 2
    maxPixels=1e13
)

# === BUG 5 is here ===
# Calculate loss area — but without masking to forest first!  # 🐛 Bug 5
buggy_loss_area = buggy_loss_decade.multiply(ee.Image.pixelArea()).divide(10000)
buggy_stats = buggy_loss_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=buggy_region,
    scale=1000,                       # Also uses wrong scale
    maxPixels=1e13
)

print("--- Buggy Results ---")
try:
    print(f"Average tree cover: {buggy_avg_cover.getInfo()}")
except Exception as e:
    print(f"Error getting tree cover stats: {e}")
    print("(This is one of the bugs!)")

try:
    print(f"Forest loss area (2010-2020): {buggy_stats.getInfo()} hectares")
except Exception as e:
    print(f"Error getting loss stats: {e}")

---
### 🐛 Bug Hints

Try to find all 5 bugs first! If you're stuck, expand the hints below.

---

**Hint 1** (vague): Check the band names against the [official dataset documentation](https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2023_v1_11). Are they spelled correctly?

---

**Hint 2** (specific): The Hansen GFC dataset has a native resolution of ~30 metres. What happens when you use `scale=1000`? You're averaging 30m pixels into 1km blocks before computing statistics — **losing massive amounts of spatial detail**.

---

**Hint 3** (specific): The comment says "forest loss between 2010 and 2020". In the Hansen dataset, `lossyear=10` means 2010 and `lossyear=20` means 2020. Does `.lte(19)` include 2020?

---

**Hint 4** (specific): What does `ee.Reducer.sum()` do to percentage values (0–100)? If tree cover is 80% in one pixel and 60% in another, does summing them give you a meaningful "average tree cover"?

---

**Hint 5** (specific): The code calculates loss area across ALL pixels — including non-forest areas, water, etc. Should you first mask to areas that were actually forest in 2000 (e.g., `treecover2000 >= 30`)?

In [ ]:
# @title Click to reveal solution for Bug 1: Wrong band name

# ❌ BUGGY:   buggy_gfc.select('tree_cover')
# ✅ CORRECT: buggy_gfc.select('treecover2000')
#
# The band name is 'treecover2000', not 'tree_cover'.
# This is a very common AI error — the model "invents" plausible
# band names that don't exist in the actual dataset.
#
# ALWAYS check band names against the official GEE Data Catalog:
# https://developers.google.com/earth-engine/datasets/

fixed_treecover = buggy_gfc.select('treecover2000')
print("✅ Bug 1 fixed: Correct band name is 'treecover2000'")

In [ ]:
# @title Click to reveal solution for Bug 2: Wrong scale

# ❌ BUGGY:   scale=1000   (1 km resolution)
# ✅ CORRECT: scale=30     (30m — native Hansen resolution)
#
# The Hansen GFC dataset has a native resolution of ~30 metres.
# Using scale=1000 tells GEE to aggregate 30m pixels into 1km blocks
# BEFORE computing statistics. This means:
#   - Small patches of forest loss are averaged away
#   - Area calculations are drastically wrong
#   - You lose ~1,000x spatial detail
#
# Rule: Use the native resolution of your input dataset,
# UNLESS you deliberately want to downsample (and document why).

# Example of correct reduceRegion call:
# stats = image.reduceRegion(
#     reducer=ee.Reducer.sum(),
#     geometry=region,
#     scale=30,           # ← Match native resolution
#     maxPixels=1e13
# )

print("✅ Bug 2 fixed: Use scale=30 to match Hansen's native resolution")

In [ ]:
# @title Click to reveal solution for Bug 3: Off-by-one date error

# ❌ BUGGY:   lossyear.gte(10).And(lossyear.lte(19))
#             This gives 2010-2019 (misses 2020!)
#
# ✅ CORRECT: lossyear.gte(10).And(lossyear.lte(20))
#             This gives 2010-2020 (includes 2020)
#
# In Hansen GFC:
#   lossyear = 10 → year 2010
#   lossyear = 19 → year 2019
#   lossyear = 20 → year 2020  ← This was excluded!
#
# Off-by-one errors are one of the most common bugs in programming.
# AI models are especially prone to them because they "know" the
# pattern but sometimes get the boundary wrong.
#
# ALWAYS verify date ranges with a simple sanity check!

fixed_lossyear = buggy_gfc.select('lossyear')
fixed_loss_decade = fixed_lossyear.gte(10).And(fixed_lossyear.lte(20))
print("✅ Bug 3 fixed: .lte(20) includes year 2020")

In [ ]:
# @title Click to reveal solution for Bug 4: Wrong reducer

# ❌ BUGGY:   ee.Reducer.sum() on tree cover percentage
#             Sum of percentages is meaningless!
#             If you have pixels with 80%, 60%, 40%, sum = 180 ???
#
# ✅ CORRECT: ee.Reducer.mean() for average tree cover percentage
#             Mean of percentages gives the average canopy cover
#
# When to use which reducer:
#   - sum()  → for AREAS (multiply a mask by pixelArea first)
#   - mean() → for AVERAGES of continuous variables (%, temperature, etc.)
#   - count()→ for counting pixels
#   - min()/max() → for extremes
#
# This is a subtle but critical error. The code runs fine,
# and the result is a number — but the number is meaningless.

fixed_avg_cover = fixed_treecover.reduceRegion(
    reducer=ee.Reducer.mean(),  # ← Correct reducer for percentages
    geometry=buggy_region,
    scale=30,
    maxPixels=1e13,
    bestEffort=True
)
print("✅ Bug 4 fixed: Use ee.Reducer.mean() for average percentage")

In [ ]:
# @title Click to reveal solution for Bug 5: Missing forest mask

# ❌ BUGGY:   Statistics computed over ALL pixels, including:
#             - Non-forest areas (grassland, urban, agriculture)
#             - Water bodies
#             - Areas with very low tree cover
#             This inflates the "forest loss" numbers.
#
# ✅ CORRECT: First mask to areas that WERE forest in 2000,
#             THEN compute loss statistics.
#
# Why this matters:
#   - The Hansen dataset records "tree cover loss" everywhere,
#     not just in forests. A single tree in a city being cut down
#     would be counted as "loss".
#   - For deforestation analysis, you typically want to focus on
#     areas that had significant forest cover (e.g., >= 30%).

# Create forest mask (areas with >= 30% tree cover in 2000)
fixed_forest_mask = fixed_treecover.gte(30)

# Apply the forest mask to the loss layer
fixed_loss_in_forest = fixed_loss_decade.updateMask(fixed_forest_mask)

# NOW compute the area
fixed_loss_area = fixed_loss_in_forest.multiply(ee.Image.pixelArea()).divide(10000)
fixed_stats = fixed_loss_area.reduceRegion(
    reducer=ee.Reducer.sum(),
    geometry=buggy_region,
    scale=30,              # ← Also fixed scale here
    maxPixels=1e13,
    bestEffort=True
)

print("✅ Bug 5 fixed: Applied forest mask (treecover2000 >= 30%) before computing loss")
print(f"\nCorrected forest loss (2010-2020): {fixed_stats.getInfo()} hectares")

### 🐛 Bug Summary

| Bug # | Type | What was wrong | Impact |
|-------|------|---------------|--------|
| 1 | **Band name** | `'tree_cover'` → `'treecover2000'` | Code crashes or returns wrong band |
| 2 | **Scale** | `scale=1000` → `scale=30` | ~1,000x loss of spatial detail |
| 3 | **Off-by-one** | `.lte(19)` → `.lte(20)` | Missing an entire year of data |
| 4 | **Wrong reducer** | `sum()` → `mean()` for percentages | Scientifically meaningless result |
| 5 | **Missing mask** | No forest mask applied | Inflated loss statistics |

### ⚠️ Teaching Point

**All of these bugs are real examples of errors that AI coding assistants produce.** The code looks reasonable, runs without Python errors, and returns numbers — but the numbers are **wrong**.

This is why **domain expertise is essential** when using AI for scientific analysis. You must:
1. **Understand the dataset** you're working with (band names, resolutions, value meanings)
2. **Verify results** against known values or published studies
3. **Question every parameter** — why this scale? Why this reducer? Why this threshold?

---

---
# Section 5: Exercise 1 — Modify for Your Region
**⏱️ ~10 minutes**

---

### Your turn!

Now that you've seen how the deforestation analysis works, **adapt it for a region you care about**. This could be:

- 🌳 **Amazon Basin** — the world's largest tropical forest
- 🌍 **Congo Basin** — Africa's great forest
- 🌏 **Southeast Asia** — rapid land-use change
- 🦎 **Madagascar** — unique biodiversity under threat
- 🏠 **Your study area** — wherever your research takes you!

### What to modify:
1. **Region coordinates** — change the bounding box
2. **Forest threshold** — 30% works for dense tropical forest, but drier forests or woodlands might need a lower threshold (e.g., 10-20%)
3. **Time period** — focus on a specific decade or recent years

In [ ]:
# ============================================================
# EXERCISE 1: Modify the deforestation analysis for YOUR region
# ============================================================

# TODO: Define your region of interest
# Replace these coordinates with your area
# Format: [longitude_min, latitude_min, longitude_max, latitude_max]
my_region = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])  # ← CHANGE THIS!

# TODO: Choose a forest cover threshold appropriate for your region
# - Dense tropical forest: 30% (default)
# - Dry woodland/savanna: 10-20%
# - Boreal forest: 25%
my_threshold = 30  # ← Modify if needed

# TODO: Choose your map centre and zoom level
my_centre = [5.8, 117.0]  # ← [latitude, longitude]
my_zoom = 8               # ← Higher = more zoomed in

# ============================================================
# The analysis pipeline (same as Worked Example 2)
# You shouldn't need to change anything below this line!
# ============================================================

# Load Hansen GFC
my_gfc = ee.Image(HANSEN_DATASET_ID).select(['treecover2000', 'loss', 'lossyear'])

# Create forest mask
my_treecover = my_gfc.select('treecover2000')
my_forest = my_treecover.gte(my_threshold)

# Loss layers
my_loss = my_gfc.select('loss')
my_lossyear = my_gfc.select('lossyear')

# Create map
MapEx1 = geemap.Map(center=my_centre, zoom=my_zoom)
MapEx1.addLayer(my_forest.selfMask(), {'palette': ['00FF00']}, 'Forest 2000', opacity=0.5)
MapEx1.addLayer(my_loss.selfMask(), {'palette': ['FF0000']}, 'Forest Loss')
MapEx1.addLayer(
    my_lossyear.selfMask(),
    {'min': 1, 'max': 23, 'palette': ['0000FF', '00FFFF', '00FF00', 'FFFF00', 'FF0000']},
    'Loss Year',
    shown=False
)

MapEx1

In [ ]:
# @title Click to reveal example coordinates for suggested regions

# Amazon Basin (Rondônia state, Brazil — deforestation frontier)
# my_region = ee.Geometry.Rectangle([-65.0, -13.0, -60.0, -8.0])
# my_centre = [-10.5, -62.5]

# Congo Basin (eastern DRC — high biodiversity)
# my_region = ee.Geometry.Rectangle([25.0, -5.0, 30.0, 2.0])
# my_centre = [-1.5, 27.5]

# Sumatra, Indonesia (oil palm expansion hotspot)
# my_region = ee.Geometry.Rectangle([100.0, -3.0, 105.0, 2.0])
# my_centre = [-0.5, 102.5]

# Madagascar (eastern rainforest corridor)
# my_region = ee.Geometry.Rectangle([46.0, -22.0, 50.0, -14.0])
# my_centre = [-18.0, 48.0]

print("Uncomment one of the regions above and re-run the exercise cell!")

---
# Section 6: Exercise 2 — Trend Analysis Challenge 📈
**⏱️ ~10 minutes**

---

### Challenge: 5-Year Moving Average of Deforestation

Calculate the **5-year moving average** of annual deforestation rate in Sabah.

**Question:** Which 5-year period had the **highest average annual forest loss**?

🏆 **Closest answer wins a prize!**

### What is a moving average?

A 5-year moving average smooths out year-to-year fluctuations by averaging each year with the 2 years before and 2 years after it. This reveals underlying trends.

For example, the moving average for 2005 = mean(2003, 2004, 2005, 2006, 2007).

In [ ]:
# ============================================================
# EXERCISE 2: Trend Analysis Challenge
# ============================================================

# --- Data loading (PROVIDED — don't modify) ---
gfc_ex2 = ee.Image(HANSEN_DATASET_ID).select(['lossyear'])
sabah_ex2 = ee.Geometry.Rectangle([115.5, 4.5, 118.5, 7.0])

# If you already computed annual_loss_ha in Section 3, you can reuse it!
# Otherwise, calculate it here.
# annual_loss_ha should be a list of 23 values (2001-2023) in hectares.

# Check if we already have the data from Section 3
try:
    _ = annual_loss_ha[0]
    print(f"✅ Reusing annual_loss_ha from Section 3 ({len(annual_loss_ha)} years)")
    ex2_years = list(range(2001, 2024))
    ex2_loss = annual_loss_ha
except NameError:
    print("⚠️ annual_loss_ha not found — you need to run Section 3 first!")
    print("   Or calculate it here using the same approach.")
    ex2_years = []
    ex2_loss = []

In [ ]:
# ============================================================
# YOUR CODE HERE: Calculate 5-year moving average
# ============================================================

# TODO: Compute 5-year moving averages
# Hint: You can do this in Python after getting the yearly values.
# A simple approach:
#   For each position i (where i >= 2 and i <= len-3),
#   moving_avg[i] = mean of values from i-2 to i+2 (inclusive)

# moving_avg = []
# for i in range(2, len(ex2_loss) - 2):
#     window = ex2_loss[i-2 : i+3]  # 5 elements
#     moving_avg.append(np.mean(window))

# TODO: Find the 5-year window with maximum average annual loss
# Hint: The centre year of the window with the highest moving average
#       tells you which 5-year period was worst.

# max_idx = np.argmax(moving_avg)
# centre_year = ex2_years[max_idx + 2]  # +2 because moving_avg starts at index 2
# print(f"Highest 5-year average centred on: {centre_year}")
# print(f"Average annual loss: {moving_avg[max_idx]:,.0f} hectares/year")

pass  # ← Remove this line when you add your code!

In [ ]:
# ============================================================
# YOUR CODE HERE: Plot annual values and moving average
# ============================================================

# TODO: Create a plot showing both:
#   1. Annual deforestation (as bars or line)
#   2. 5-year moving average (as a smooth line)

# Hint: matplotlib example
# fig, ax = plt.subplots(figsize=(14, 6))
# ax.bar(ex2_years, ex2_loss, alpha=0.5, label='Annual loss')
# ax.plot(ex2_years[2:-2], moving_avg, color='red', linewidth=2,
#         label='5-year moving average', marker='o')
# ax.set_xlabel('Year')
# ax.set_ylabel('Forest Loss (hectares)')
# ax.set_title('Annual Deforestation & 5-Year Moving Average — Sabah')
# ax.legend()
# plt.tight_layout()
# plt.show()

pass  # ← Remove this line when you add your code!

In [ ]:
# @title Click to reveal the full solution for Exercise 2

# ============================================================
# SOLUTION: 5-year moving average of deforestation
# ============================================================

if len(ex2_loss) > 0:
    # Calculate 5-year moving average
    window_size = 5
    moving_avg = []
    for i in range(2, len(ex2_loss) - 2):
        window = ex2_loss[i-2 : i+3]  # 5 elements centred on i
        moving_avg.append(np.mean(window))

    # The years corresponding to the moving average centres
    ma_years = ex2_years[2:-2]

    # Find the 5-year window with maximum average
    max_idx = np.argmax(moving_avg)
    peak_centre_year = ma_years[max_idx]
    peak_window = f"{peak_centre_year - 2}–{peak_centre_year + 2}"

    print(f"🏆 Highest 5-year average deforestation period:")
    print(f"   Window: {peak_window}")
    print(f"   Centre year: {peak_centre_year}")
    print(f"   Average annual loss: {moving_avg[max_idx]:,.0f} hectares/year")

    # Plot
    fig, ax = plt.subplots(figsize=(14, 6))

    # Annual values as bars
    ax.bar(ex2_years, ex2_loss, alpha=0.4, color='coral', label='Annual loss', edgecolor='darkred', linewidth=0.5)

    # 5-year moving average as a line
    ax.plot(ma_years, moving_avg, color='darkred', linewidth=2.5,
            label=f'5-year moving average', marker='o', markersize=4, zorder=5)

    # Highlight the peak window
    ax.axvspan(peak_centre_year - 2, peak_centre_year + 2, alpha=0.1, color='red',
               label=f'Peak period ({peak_window})')

    ax.set_xlabel('Year', fontsize=12)
    ax.set_ylabel('Forest Loss (hectares)', fontsize=12)
    ax.set_title('Annual Deforestation & 5-Year Moving Average — Sabah, Malaysian Borneo', fontsize=14)
    ax.legend(fontsize=11)
    ax.grid(axis='y', alpha=0.3)
    ax.set_axisbelow(True)
    ax.get_yaxis().set_major_formatter(plt.FuncFormatter(lambda x, p: f'{x:,.0f}'))

    plt.tight_layout()
    plt.show()
else:
    print("⚠️ No data available. Run Section 3 first!")

---
# Section 7: Exercise 3 — Open Innovation 🚀
**⏱️ If time permits**

---

### Impress us!

Use AI to build something we haven't shown you. This is your chance to explore and experiment. Here are some ideas:

- 🔥 **Fire monitoring** — Use [FIRMS active fire data](https://developers.google.com/earth-engine/datasets/catalog/FIRMS) to map recent wildfires
- 🌱 **NDVI trends** — Calculate Normalised Difference Vegetation Index trends over time using Landsat or MODIS
- 🏙️ **Urban expansion** — Map how cities have grown using nighttime lights (VIIRS) or land cover datasets
- 💧 **Water body change** — Track shrinking lakes or expanding reservoirs using the JRC Global Surface Water dataset
- 🌊 **Mangrove loss** — Use the Global Mangrove Watch or Hansen data to assess mangrove deforestation
- 🌡️ **Surface temperature** — Map land surface temperature patterns using MODIS LST data

### Tips for working with AI assistants:

1. **Be specific** in your prompts — include the dataset name, region, time period
2. **Ask for GEE Python code** specifically (not JavaScript)
3. **Verify band names** against the GEE Data Catalog
4. **Check the scale** — what resolution is the dataset?
5. **Review the output** — does it make physical sense?

In [ ]:
# ============================================================
# Your innovation here!
# ============================================================
#
# Use this cell (and add more cells below) to build your own
# Earth Observation analysis. Use AI to help — but remember to
# verify everything!
#
# Some useful GEE dataset IDs to get you started:
#   FIRMS active fires:      'FIRMS'
#   MODIS NDVI:              'MODIS/061/MOD13A2'
#   VIIRS nighttime lights:  'NOAA/VIIRS/DNB/MONTHLY_V1/VCMSLCFG'
#   JRC surface water:       'JRC/GSW1_4/GlobalSurfaceWater'
#   MODIS land surface temp: 'MODIS/061/MOD11A1'
#
# Happy coding! 🎉

pass

---
# Section 8: Wrap-up 🎓

---

## What You've Learned Today

In this workshop, you've:

1. ✅ **Set up** a reproducible GEE analysis environment
2. ✅ **Visualised** satellite imagery at regional scales
3. ✅ **Analysed** 23 years of deforestation data
4. ✅ **Debugged** AI-generated code with 5 hidden bugs
5. ✅ **Adapted** analysis pipelines to new regions
6. ✅ **Calculated** trend statistics from Earth Observation data

---

## The Six Principles of Responsible AI-Assisted EO

| # | Principle | What it means |
|---|-----------|---------------|
| 1 | **Understand** | Know what every line of code does before running it |
| 2 | **Validate** | Check results against known data or published values |
| 3 | **Document** | Record software versions, dataset IDs, and AI models used |
| 4 | **Reproduce** | Ensure your analysis can be re-run and gives the same results |
| 5 | **Verify** | Check parameters (scale, bands, dates, thresholds) against source documentation |
| 6 | **Attribute** | Credit datasets, tools, and AI assistance in your publications |

---

## 📋 Printable Checklist

Use this checklist every time you write GEE code with AI assistance:

- [ ] Band names match the official GEE Data Catalog
- [ ] Scale/resolution matches the native dataset resolution
- [ ] Date ranges are correct (watch for off-by-one errors!)
- [ ] Reducers are appropriate (sum for areas, mean for averages)
- [ ] Masks are applied (forest, land, cloud, quality)
- [ ] No unnecessary `.clip()` (use `region` parameter instead)
- [ ] No unnecessary `.reproject()` (let the export/display control projection)
- [ ] `.select()` is called early to reduce memory
- [ ] `.filterBounds()` and `.filterDate()` are applied early
- [ ] Results pass a sanity check (reasonable magnitude? correct units?)
- [ ] Reproducibility metadata is recorded

---

## 📚 Further Resources

### Google Earth Engine
- [GEE Developer Guide](https://developers.google.com/earth-engine)
- [GEE Data Catalog](https://developers.google.com/earth-engine/datasets)
- [GEE Python API Reference](https://developers.google.com/earth-engine/apidocs)
- [geemap documentation](https://geemap.org/)

### Datasets used in this workshop
- [Hansen Global Forest Change](https://developers.google.com/earth-engine/datasets/catalog/UMD_hansen_global_forest_change_2023_v1_11) — Hansen et al. (2013) *Science*
- [Landsat 8 Collection 2 SR](https://developers.google.com/earth-engine/datasets/catalog/LANDSAT_LC08_C02_T1_L2)
- [World Database on Protected Areas (WDPA)](https://developers.google.com/earth-engine/datasets/catalog/WCMC_WDPA_current_polygons)

### Responsible AI in science
- [Nature: AI and the future of scientific publishing](https://www.nature.com/articles/d41586-023-00056-7)
- [Google AI Principles](https://ai.google/principles/)

---

## 💾 Save This Notebook!

Don't forget to save this notebook to your Google Drive:

**File → Save a copy in Drive**

This ensures you keep your work, including any modifications you made during the exercises.

---

### Thank you for attending! 🌍

*Good Vibes Workshop — ECCB 2026*  
*Institute of Zoology, Zoological Society of London (ZSL)*  
*AB, HFT, HH, JW*

📄 This notebook is released under [CC BY 4.0](https://creativecommons.org/licenses/by/4.0/).